<a href="https://colab.research.google.com/github/haida-ishtiaq/FlyRankAI-ML-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haida-ishtiaq/FlyRankAI-ML-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task Type: Ranking and Opportunity Scoring (supported by binary classification)**

**Why this task type:**
Content operations teams managing thousands of articles across 32 clients do not have the bandwidth to review or rewrite every page in their catalog. The core business decision is not simply asking *"is this page declining?"* (binary classification), but answering: **"Which 50 or 100 pages should our editorial team refresh THIS WEEK to maximize organic traffic recovery and editorial ROI?"**

This makes the problem fundamentally a **Ranking & Priority Scoring** task:
1. **Scoring:** Assigning an estimated opportunity score to each content item reflecting its expected traffic/impression loss if left unhandled vs potential recovery upon refresh.
2. **Ranking:** Sorting all active content items within a client portfolio by this score to produce an actionable, ordered queue for editors.

In [15]:
import os
import pandas as pd
import numpy as np

if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    os.chdir("/content")
    if not os.path.isdir("FlyRankAI-ML-Internship"):
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                         "https://github.com/haida-ishtiaq/FlyRankAI-ML-Internship"], check=True)
    os.chdir("FlyRankAI-ML-Internship")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("=== 1. LANE & SCALE AUDIT ===")
print(f"Lane: Refresh / Content Opportunity Scoring")
print(f"Task Type: Priority Ranking & Scoring")
print(f"Total Content Items: {len(df):,}")
print(f"Unique Clients: {df['client_id'].nunique()}")
print(f"Average Pages per Client: {len(df) / df['client_id'].nunique():.1f}")
print(f"Weekly Review Capacity (Estimated @ 50 pages/client/week): {df['client_id'].nunique() * 50:,} pages")
print(f"Capacity Gap: Editorial team can only review ~5.3% of catalog per week — ranking is essential!")


=== 1. LANE & SCALE AUDIT ===
Lane: Refresh / Content Opportunity Scoring
Task Type: Priority Ranking & Scoring
Total Content Items: 30,000
Unique Clients: 32
Average Pages per Client: 937.5
Weekly Review Capacity (Estimated @ 50 pages/client/week): 1,600 pages
Capacity Gap: Editorial team can only review ~5.3% of catalog per week — ranking is essential!



## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*
Target Definition: is_declining — a genuinely observed outcome, per framing-ml-problems/SKILL.md's core rule that "the target must be observed, not defined."

is_declining = (trend_direction == "down"). Per flyrank-data/SKILL.md, trend_direction is computed upstream from trend_pct, a measured 30-day-over-30-day impression trend — not a threshold I invented. This is chosen here, at the framing stage, on that basis alone.

(Forward-looking note, not part of the justification: every later notebook in this repo — w04_baseline_score.ipynb, w05_model.ipynb onward — builds on this same target, so the definition set here carries through the whole project without needing to be redefined later.)


In [16]:
# --- The actual target: a genuinely observed outcome ---
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

pos_count = df['is_declining'].sum()
total_count = len(df)
base_rate = (pos_count / total_count) * 100

print("=== 2. TARGET VERIFICATION (observed outcome) ===")
print(f"Target Label Name: is_declining")
print(f"Target Definition: trend_direction == 'down' (observed 30d-vs-30d trend, computed upstream from trend_pct)")
print(f"Positive (declining): {pos_count:,} / {total_count:,} ({base_rate:.2f}%)")
print(f"Negative (not declining): {total_count - pos_count:,} ({100 - base_rate:.2f}%)")

# --- The eligibility filter, kept separate and honestly labeled as a defined rule ---
df['impressions_drop'] = df['impressions_prev_30d'] - df['impressions_last_30d']
df['is_eligible_for_queue'] = (
    (df['impressions_90d'] >= 500) &
    (df['content_age_days'] >= 180)
).astype(int)

print("\n=== ELIGIBILITY FILTER (a defined rule -- NOT the prediction target) ===")
print(f"Filter: impressions_90d >= 500 AND content_age_days >= 180")
print(f"Eligible pages: {df['is_eligible_for_queue'].sum():,} / {total_count:,} ({df['is_eligible_for_queue'].mean():.1%})")
print("This filter decides which pages are worth ranking at all -- it does not predict anything,")
print("and is never confused with the observed target 'is_declining' above.")

=== 2. TARGET VERIFICATION (observed outcome) ===
Target Label Name: is_declining
Target Definition: trend_direction == 'down' (observed 30d-vs-30d trend, computed upstream from trend_pct)
Positive (declining): 16,262 / 30,000 (54.21%)
Negative (not declining): 13,738 (45.79%)

=== ELIGIBILITY FILTER (a defined rule -- NOT the prediction target) ===
Filter: impressions_90d >= 500 AND content_age_days >= 180
Eligible pages: 9,929 / 30,000 (33.1%)
This filter decides which pages are worth ranking at all -- it does not predict anything,
and is never confused with the observed target 'is_declining' above.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary Metric: Precision@K (specifically Precision@50 and Precision@100), computed against `is_declining`.**

**Why Precision@K is the defendable metric:**
- **Editorial Capacity Alignment:** every false positive in a top-K queue directly wastes editorial time.
- **Why plain accuracy is incomplete:** a model can score well on accuracy by predicting the majority class everywhere, while being useless at the top of the queue. Precision@K evaluates exactly where decisions happen.

**Baselines to beat, computed on the real target (`is_declining`), not the old proxy:**

In [17]:
def precision_at_k(df_data, score_col, target_col='is_declining', k=100):
    top_k = df_data.sort_values(by=score_col, ascending=False).head(k)
    return top_k[target_col].mean()

p50_impr = precision_at_k(df, 'impressions_90d', k=50)
p100_impr = precision_at_k(df, 'impressions_90d', k=100)
p50_drop = precision_at_k(df, 'impressions_drop', k=50)
p100_drop = precision_at_k(df, 'impressions_drop', k=100)
base_rate_frac = df['is_declining'].mean()

print("=== 3. SUCCESS METRIC BENCHMARKS (against is_declining) ===")
print(f"Base rate (random-order expectation): {base_rate_frac:.2%}")
print(f"Baseline (sort by 90d impressions): Precision@50 = {p50_impr:.2%}, Precision@100 = {p100_impr:.2%}")
print(f"Baseline (sort by 30d impression drop): Precision@50 = {p50_drop:.2%}, Precision@100 = {p100_drop:.2%}")
print(f"Target success goal for a trained model: Precision@50 clearly above both baselines AND the base rate")
print("(this replaces the earlier fixed 75% target, which was set before any baseline was actually measured")
print("against the real target -- per framing-ml-problems/SKILL.md: 'name the metric before training,")
print("good defined after the fact always looks good' -- the goal here is comparative, not an arbitrary number.)")

=== 3. SUCCESS METRIC BENCHMARKS (against is_declining) ===
Base rate (random-order expectation): 54.21%
Baseline (sort by 90d impressions): Precision@50 = 42.00%, Precision@100 = 38.00%
Baseline (sort by 30d impression drop): Precision@50 = 100.00%, Precision@100 = 96.00%
Target success goal for a trained model: Precision@50 clearly above both baselines AND the base rate
(this replaces the earlier fixed 75% target, which was set before any baseline was actually measured
against the real target -- per framing-ml-problems/SKILL.md: 'name the metric before training,
good defined after the fact always looks good' -- the goal here is comparative, not an arbitrary number.)


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of Analysis: One row = one pseudonymized content item (`content_id`) within a client (`client_id`), over a trailing 90-day observation window.**

- **Grain:** `content_id` (unique per row; `client_id` groups rows for splitting, never a per-row identifier of the unit itself).
- **Target:** `is_declining` (Section 2).
- **Eligibility filter:** `is_eligible_for_queue` (Section 2) — shown alongside the target here so the difference between "predicted outcome" and "included in the queue at all" is visible in the same preview, not just described in prose.

In [18]:
analysis_cols = [
    'content_id', 'content_type', 'content_age_days',
    'impressions_90d', 'avg_position', 'ctr',
    'is_eligible_for_queue', 'is_declining'
]

print("=== 4. UNIT OF ANALYSIS DATAFRAME PREVIEW ===")
print(f"DataFrame Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Unit of Analysis Grain: content_id (client_id used only for grouped splitting)")
print("\nSample Slice (Top 5 rows, client_id excluded from display per the self-check rule):")
print(df[analysis_cols].head().to_string(index=False))

=== 4. UNIT OF ANALYSIS DATAFRAME PREVIEW ===
DataFrame Shape: 30,000 rows x 47 columns
Unit of Analysis Grain: content_id (client_id used only for grouped splitting)

Sample Slice (Top 5 rows, client_id excluded from display per the self-check rule):
          content_id    content_type  content_age_days  impressions_90d  avg_position  ctr  is_eligible_for_queue  is_declining
content_304f48230142 keyword article               187             3803          10.6 0.76                      1             1
content_a1fb4e703a9e keyword article               445            15320          20.3 0.05                      1             1
content_9aa793d4d895 keyword article               141            12581          36.5 0.09                      0             1
content_331d6c4de07b keyword article               463            11751           6.2 0.49                      1             0
content_d99b7a2d90ca keyword article               263            19140          44.0 0.13                  

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Why a plain rule fails, tested against the real target this time:**

In [19]:
rule_flagged = df['content_age_days'] >= 180
total_rule_flagged = rule_flagged.sum()
rule_true_positives = (df[rule_flagged]['is_declining'] == 1).sum()
rule_false_positives = total_rule_flagged - rule_true_positives
rule_precision = rule_true_positives / total_rule_flagged

print("=== 5. FIXED RULE VS ML COMPARISON (against is_declining) ===")
print(f"Fixed Rule Tested: content_age_days >= 180")
print(f"Total Pages Flagged: {total_rule_flagged:,} ({rule_flagged.mean():.1%} of catalog)")
print(f"True positives (actually declining): {rule_true_positives:,}")
print(f"False positives (wasted effort): {rule_false_positives:,}")
print(f"Rule precision: {rule_precision:.2%}")
print(f"(For reference, base rate is {df['is_declining'].mean():.2%} -- compare the rule's precision to this,")
print(f"not to zero, since some baseline hit rate is expected from the population itself.)")
print("\nConclusion: a single-threshold rule mixes true and false positives without regard to position,")
print("CTR, or visibility -- signals that interact non-linearly and that a model can learn jointly.")

=== 5. FIXED RULE VS ML COMPARISON (against is_declining) ===
Fixed Rule Tested: content_age_days >= 180
Total Pages Flagged: 17,986 (60.0% of catalog)
True positives (actually declining): 8,739
False positives (wasted effort): 9,247
Rule precision: 48.59%
(For reference, base rate is 54.21% -- compare the rule's precision to this,
not to zero, since some baseline hit rate is expected from the population itself.)

Conclusion: a single-threshold rule mixes true and false positives without regard to position,
CTR, or visibility -- signals that interact non-linearly and that a model can learn jointly.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.